In [ ]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

# Setup CSV Database File Name
DB_FILE = 'lost_and_found_db.csv'

# Initialize CSV if it doesn't exist
if not os.path.exists(DB_FILE):
    df_init = pd.DataFrame(columns=[
        "ID", "Type", "Item Name", "Category", "Location", "Date", "Description", "Contact Info", "Status"
    ])
    df_init.to_csv(DB_FILE, index=False)

def load_data():
    return pd.read_csv(DB_FILE)

def save_data(df):
    df.to_csv(DB_FILE, index=False)

In [ ]:
# Create Widgets for UI
out = widgets.Output()

# Form Inputs
item_type = widgets.Dropdown(options=['Lost', 'Found'], description='Type:')
item_name = widgets.Text(placeholder='e.g., iPhone 13, Leather Wallet', description='Item Name:')
category = widgets.Dropdown(options=['Electronics', 'Personal Belongings', 'Clothing', 'Documents/Cards', 'Keys', 'Other'], description='Category:')
location = widgets.Text(placeholder='e.g., Library 2nd Floor, Cafeteria', description='Location:')
description = widgets.Textarea(placeholder='Color, brand, distinguishing features...', description='Description:')
contact_info = widgets.Text(placeholder='Email or Phone Number', description='Contact:')
submit_btn = widgets.Button(description='Submit Report', button_style='success', icon='check')

# Search Inputs
search_keyword = widgets.Text(placeholder='Search by item name, location...', description='Keyword:')
filter_type = widgets.Dropdown(options=['All', 'Lost', 'Found'], description='Filter Type:')
filter_status = widgets.Dropdown(options=['Open', 'Claimed/Resolved', 'All'], description='Status:')
search_btn = widgets.Button(description='Search Items', button_style='info', icon='search')

# Resolve Inputs
resolve_id = widgets.BoundedIntText(value=1, min=1, max=10000, step=1, description='Item ID:')
resolve_btn = widgets.Button(description='Mark as Resolved', button_style='warning', icon='flag')

# Function to Add New Record
def on_submit_click(b):
    with out:
        clear_output()
        if not item_name.value or not location.value or not contact_info.value:
            print("❌ Please fill in Item Name, Location, and Contact Information.")
            return

        df = load_data()
        new_id = len(df) + 1 if len(df) == 0 else df['ID'].max() + 1
        current_date = datetime.now().strftime("%Y-%m-%d %H:%M")

        new_entry = {
            "ID": new_id,
            "Type": item_type.value,
            "Item Name": item_name.value,
            "Category": category.value,
            "Location": location.value,
            "Date": current_date,
            "Description": description.value,
            "Contact Info": contact_info.value,
            "Status": "Open"
        }

        df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=False)
        save_data(df)
        print(f"✅ Successfully registered item! Item ID assigned: {new_id}")

        # Reset fields
        item_name.value = ''
        location.value = ''
        description.value = ''
        contact_info.value = ''

# Function to Search Records
def on_search_click(b):
    with out:
        clear_output()
        df = load_data()

        if filter_type.value != 'All':
            df = df[df['Type'] == filter_type.value]

        if filter_status.value != 'All':
            df = df[df['Status'] == filter_status.value]

        if search_keyword.value:
            query = search_keyword.value.lower()
            df = df[
                df['Item Name'].str.lower().str.contains(query, na=False) |
                df['Location'].str.lower().str.contains(query, na=False) |
                df['Description'].str.lower().str.contains(query, na=False)
            ]

        if len(df) == 0:
            print("ℹ️ No records found matching your query.")
        else:
            display(df)

# Function to Resolve Item
def on_resolve_click(b):
    with out:
        clear_output()
        df = load_data()
        target_id = resolve_id.value

        if target_id in df['ID'].values:
            df.loc[df['ID'] == target_id, 'Status'] = 'Claimed/Resolved'
            save_data(df)
            print(f"🎉 Item ID {target_id} has been marked as Claimed/Resolved!")
        else:
            print(f"❌ Item ID {target_id} does not exist in the database.")

# Bind Event Handlers
submit_btn.on_click(on_submit_click)
search_btn.on_click(on_search_click)
resolve_btn.on_click(on_resolve_click)

# Design Layout with Tabs
form_box = widgets.VBox([
    item_type, item_name, category, location, description, contact_info, submit_btn
])

search_box = widgets.VBox([
    widgets.HBox([filter_type, filter_status]),
    widgets.HBox([search_keyword, search_btn])
])

resolve_box = widgets.VBox([
    widgets.HTML(value="**Enter the Item ID to mark it as found or returned:**"),
    widgets.HBox([resolve_id, resolve_btn])
])

tab = widgets.Tab()
tab.children = [form_box, search_box, resolve_box]
tab.set_title(0, 'Report Item')
tab.set_title(1, 'Search Database')
tab.set_title(2, 'Resolve/Claim')

# Display UI
print("=== DIGITAL LOST AND FOUND SYSTEM ===")
display(tab, out)

=== DIGITAL LOST AND FOUND SYSTEM ===


Output()